# 01 Data Simulation: SPX Option Delta Arrivals

This notebook builds the synthetic option-trade tape required by the intraday delta-hedging project. It follows the instructions in `docs/`:

- Use student-accessible market data only.
- Cover January 1, 2026 through May 29, 2026, subject to available regular-session market bars.
- Generate synthetic listed-option trades across standard monthly expiries, strikes, maturities, trade sizes, sides, and times of day.
- Use Bloomberg **1M BVOL** as the volatility input and Bloomberg SOFR OIS **`USOSFRA Curncy`** as the 1M rate input.
- Save outputs that downstream hedging notebooks can use for static-band, TWAP/VWAP, execution-cost, and delta-risk analysis.

## Explicit Simulation Assumptions

1. The listed-option book is represented by synthetic **SPX index options**. SPX 5-minute bars provide the option spot.
2. Raw intraday timestamps start at `06:30`, so they are treated as Pacific time exports of the U.S. cash session and shifted by three hours to create `timestamp_et`.
3. 1M BVOL is used as a flat implied-volatility input across strikes and maturities. This keeps the requested 1M BVOL dependency direct and auditable.
4. `USOSFRA Curncy` is used as the 1M continuously compounded risk-free rate after converting Bloomberg percent quotes to decimals.
5. Synthetic trade times are sampled from observed SPX 5-minute bars with weights proportional to SPY volume when available; this creates open/close concentration from market data rather than a hard-coded U-shape.
6. Synthetic contract sizes are sampled from the Databento options TBBO/trade `size` field, clipped to 1-500 contracts. This is used only as an accessible proxy because actual JPM option executions are not available.
7. Buy/Sell and Call/Put are sampled 50/50, consistent with the project one-pager's allowed even split.
8. SPY and ES prices/volumes/ADV are carried into the output for later hedge-instrument comparison, but this notebook does not choose a hedge strategy.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

SEED = 20260617
N_TRADES = 50_000
START_DATE = pd.Timestamp('2026-01-01')
END_DATE = pd.Timestamp('2026-05-29')
SOFR_TICKER = 'USOSFRA Curncy'
BVOL_COLUMN = '1M BVOL'
CONTRACT_MULTIPLIER = 100

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data' / 'raw').exists():
    raise FileNotFoundError('Run this notebook from the project root or notebooks directory.')

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SIM_DIR = PROJECT_ROOT / 'data' / 'simulated'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SIM_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TRADES = SIM_DIR / 'synthetic_option_trades_2026.csv'
OUTPUT_MARKET_INPUTS = PROCESSED_DIR / 'market_inputs_2026.csv'
OUTPUT_BVOL_CLEAN = PROCESSED_DIR / 'spx_bvol_1m_clean.csv'
OUTPUT_SOFR_CLEAN = PROCESSED_DIR / 'sofr_ois_1m_clean.csv'

rng = np.random.default_rng(SEED)

## 1. Clean EOD BVOL and SOFR Inputs

The BVOL extract is in reverse chronological order and contains missing holiday rows. The cleaning rule requested here is:

1. Sort dates into chronological order.
2. Convert BVOL values to numeric.
3. Fill each missing value with the mean of the nearest previous and next available values.

For SOFR, the Bloomberg file has metadata rows. The required ticker is the **`USOSFRA Curncy`** column, which is selected explicitly by name and converted from percent to decimal form.

In [ ]:
def midpoint_fill(series: pd.Series) -> pd.Series:
    """Fill missing values with the average of previous and next valid observations."""
    numeric = pd.to_numeric(series, errors='coerce')
    prev_value = numeric.ffill()
    next_value = numeric.bfill()
    return numeric.fillna((prev_value + next_value) / 2)


def clean_bvol(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    clean = raw.rename(columns={'Date': 'date'}).copy()
    clean['date'] = pd.to_datetime(clean['date'])
    clean = clean.sort_values('date').reset_index(drop=True)

    value_columns = [c for c in clean.columns if c != 'date']
    for column in value_columns:
        clean[column] = midpoint_fill(clean[column])

    clean['bvol_1m'] = clean[BVOL_COLUMN] / 100.0
    clean['hist_vol_60d'] = clean['Hist Vol (60)'] / 100.0
    return clean[['date', BVOL_COLUMN, 'bvol_1m', 'Hist Vol (60)', 'hist_vol_60d']]


def clean_sofr(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    if SOFR_TICKER not in raw.columns:
        raise KeyError(f'{SOFR_TICKER} was not found in {path.name}. Available columns: {list(raw.columns)}')

    clean = raw.iloc[2:].rename(columns={raw.columns[0]: 'date'}).copy()
    clean['date'] = pd.to_datetime(clean['date'])
    clean['sofr_1m_pct'] = pd.to_numeric(clean[SOFR_TICKER], errors='coerce')
    clean['sofr_1m_rate'] = midpoint_fill(clean['sofr_1m_pct']) / 100.0
    clean = clean.sort_values('date').reset_index(drop=True)
    return clean[['date', 'sofr_1m_pct', 'sofr_1m_rate']]


bvol = clean_bvol(RAW_DIR / 'SPX_BVOL_extracted_table.csv')
sofr = clean_sofr(RAW_DIR / 'SOFR OIS.csv')

bvol.to_csv(OUTPUT_BVOL_CLEAN, index=False)
sofr.to_csv(OUTPUT_SOFR_CLEAN, index=False)

print(f'BVOL rows: {len(bvol):,}; date range: {bvol.date.min().date()} to {bvol.date.max().date()}')
print(f'SOFR rows: {len(sofr):,}; date range: {sofr.date.min().date()} to {sofr.date.max().date()}')
print(f'Cleaned BVOL written to {OUTPUT_BVOL_CLEAN.relative_to(PROJECT_ROOT)}')
print(f'Cleaned SOFR written to {OUTPUT_SOFR_CLEAN.relative_to(PROJECT_ROOT)}')

bvol.head(), bvol[bvol[BVOL_COLUMN].isna()]

## 2. Load 5-Minute Market Data

SPX provides option spot prices. SPY and ES are merged onto the same five-minute grid because the project later compares SPY shares and ES futures as hedge instruments. SPY volume is also used to weight synthetic option-arrival times.

In [ ]:
def load_5min_market(path: Path, symbol: str) -> pd.DataFrame:
    raw = pd.read_csv(path)
    raw = raw.rename(columns={raw.columns[0]: 'timestamp_raw'})
    raw['timestamp_raw'] = pd.to_datetime(raw['timestamp_raw'])
    raw['timestamp_et'] = raw['timestamp_raw'] + pd.Timedelta(hours=3)
    raw['trade_date'] = raw['timestamp_et'].dt.normalize()

    clean = raw.rename(columns={
        'OPEN': f'{symbol.lower()}_open',
        'HIGH': f'{symbol.lower()}_high',
        'LOW': f'{symbol.lower()}_low',
        'LAST_PRICE': f'{symbol.lower()}_price',
        'VOLUME': f'{symbol.lower()}_volume_5m',
    })
    return clean[['timestamp_raw', 'timestamp_et', 'trade_date',
                  f'{symbol.lower()}_open', f'{symbol.lower()}_high', f'{symbol.lower()}_low',
                  f'{symbol.lower()}_price', f'{symbol.lower()}_volume_5m']]


def add_daily_adv(df: pd.DataFrame, symbol: str, lookback: int = 20) -> pd.DataFrame:
    volume_col = f'{symbol.lower()}_volume_5m'
    adv_col = f'{symbol.lower()}_adv_{lookback}d'
    daily_volume = df.groupby('trade_date', as_index=False)[volume_col].sum()
    daily_volume[adv_col] = daily_volume[volume_col].rolling(lookback, min_periods=1).mean().shift(1)
    daily_volume[adv_col] = daily_volume[adv_col].bfill()
    return df.merge(daily_volume[['trade_date', adv_col]], on='trade_date', how='left')


spx = load_5min_market(RAW_DIR / 'SPX_5min.csv', 'SPX')
spy = add_daily_adv(load_5min_market(RAW_DIR / 'SPY_5min.csv', 'SPY'), 'SPY')
es1 = add_daily_adv(load_5min_market(RAW_DIR / 'ES1_5min.csv', 'ES1'), 'ES1')

market = spx.merge(
    spy.drop(columns=['timestamp_raw', 'trade_date']),
    on='timestamp_et',
    how='left',
).merge(
    es1.drop(columns=['timestamp_raw', 'trade_date']),
    on='timestamp_et',
    how='left',
)

market = market.merge(bvol[['date', 'bvol_1m']], left_on='trade_date', right_on='date', how='left').drop(columns='date')
market = market.merge(sofr[['date', 'sofr_1m_rate']], left_on='trade_date', right_on='date', how='left').drop(columns='date')
market[['bvol_1m', 'sofr_1m_rate']] = market[['bvol_1m', 'sofr_1m_rate']].ffill().bfill()

for column in ['spy_price', 'spy_volume_5m', 'spy_adv_20d', 'es1_price', 'es1_volume_5m', 'es1_adv_20d']:
    market[column] = pd.to_numeric(market[column], errors='coerce').ffill().bfill()

market = market[(market['trade_date'] >= START_DATE) & (market['trade_date'] <= END_DATE)].reset_index(drop=True)
market.to_csv(OUTPUT_MARKET_INPUTS, index=False)

print(f'Market rows: {len(market):,}; trade dates: {market.trade_date.nunique():,}')
print(f'Market inputs written to {OUTPUT_MARKET_INPUTS.relative_to(PROJECT_ROOT)}')
market.head()

## 3. Simulation Configuration

Expiries are standard monthly third Fridays. Maturity and moneyness buckets create broad coverage for later hedging tests. Time-of-day probabilities are estimated from SPY volume by 5-minute bar, and contract sizes are sampled from accessible Databento option prints.

In [ ]:
def third_friday(year: int, month: int) -> pd.Timestamp:
    first = pd.Timestamp(year=year, month=month, day=1)
    first_friday = first + pd.Timedelta(days=(4 - first.weekday()) % 7)
    return first_friday + pd.Timedelta(days=14)


def monthly_expiries(start: str = '2026-01-01', end: str = '2027-06-30') -> pd.DatetimeIndex:
    months = pd.period_range(start, end, freq='M')
    return pd.DatetimeIndex([third_friday(period.year, period.month) for period in months])


EXPIRIES = monthly_expiries()
assert all(d.weekday() == 4 and 15 <= d.day <= 21 for d in EXPIRIES)

MATURITY_BUCKETS = pd.DataFrame({
    'maturity_bucket': ['7-30d', '31-60d', '61-120d', '121-240d', '241-420d'],
    'min_dte': [7, 31, 61, 121, 241],
    'max_dte': [30, 60, 120, 240, 420],
    'weight': [0.32, 0.27, 0.21, 0.14, 0.06],
}).set_index('maturity_bucket')

MONEYNESS_BUCKETS = pd.DataFrame({
    'moneyness_bucket': ['deep_put', 'put_wing', 'near_atm', 'call_wing', 'deep_call'],
    'low': [0.80, 0.90, 0.97, 1.03, 1.10],
    'high': [0.90, 0.97, 1.03, 1.10, 1.20],
    'weight': [0.08, 0.21, 0.42, 0.21, 0.08],
}).set_index('moneyness_bucket')

TIME_BUCKETS = pd.DataFrame({
    'time_segment': ['open', 'morning', 'midday', 'afternoon', 'close'],
    'start_time': ['09:30', '10:30', '11:30', '13:30', '15:00'],
    'end_time': ['10:30', '11:30', '13:30', '15:00', '16:00'],
}).set_index('time_segment')

print(EXPIRIES[:6])
MATURITY_BUCKETS

In [ ]:
def load_contract_size_pool(path: Path) -> np.ndarray:
    raw = pd.read_csv(path, usecols=['size'])
    sizes = pd.to_numeric(raw['size'], errors='coerce').dropna().astype(int)
    sizes = sizes[(sizes > 0) & (sizes <= 500)]
    if sizes.empty:
        return np.array([1, 2, 3, 5, 10, 25, 50, 100], dtype=int)
    return sizes.to_numpy(dtype=int)


def time_segment(timestamp_et: pd.Timestamp) -> str:
    time_text = timestamp_et.strftime('%H:%M')
    for segment, row in TIME_BUCKETS.iterrows():
        if row.start_time <= time_text < row.end_time:
            return segment
    return 'close'


contract_size_pool = load_contract_size_pool(RAW_DIR / 'glbx-mdp3-20260601.tbbo.csv')
bar_weights = market['spy_volume_5m'].clip(lower=0).fillna(0).to_numpy(dtype=float)
if bar_weights.sum() == 0:
    bar_weights = np.ones(len(market), dtype=float)
bar_weights = bar_weights / bar_weights.sum()

size_summary = pd.Series(contract_size_pool).describe(percentiles=[0.5, 0.9, 0.99])
time_weight_summary = market.assign(time_segment=market['timestamp_et'].map(time_segment)).groupby('time_segment')['spy_volume_5m'].sum()
time_weight_summary = time_weight_summary / time_weight_summary.sum()

print('Databento contract-size proxy summary:')
print(size_summary.to_string())
print('\nSPY-volume time weights:')
print(time_weight_summary.reindex(TIME_BUCKETS.index).round(4).to_string())

## 4. Option Pricing and Trade Generation

Each synthetic option trade is priced with Black-Scholes using the SPX 5-minute spot, that day's cleaned 1M BVOL, and that day's cleaned `USOSFRA Curncy` 1M SOFR rate.

In [ ]:
def choose_expiry(trade_date: pd.Timestamp, bucket: str) -> pd.Timestamp:
    spec = MATURITY_BUCKETS.loc[bucket]
    dtes = (EXPIRIES - trade_date).days
    valid = (dtes >= spec.min_dte) & (dtes <= spec.max_dte)
    candidates = EXPIRIES[valid]
    candidate_dtes = dtes[valid]
    if len(candidates) == 0:
        candidates = EXPIRIES[dtes > 0]
        candidate_dtes = dtes[dtes > 0]
    midpoint = (spec.min_dte + spec.max_dte) / 2
    weights = np.asarray(np.exp(-np.abs(candidate_dtes - midpoint) / 35), dtype=float)
    return pd.Timestamp(rng.choice(candidates, p=weights / weights.sum()))


def strike_increment(spot: float) -> float:
    if spot < 500:
        return 5.0
    if spot < 2_000:
        return 10.0
    return 25.0


def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black_scholes(spot: float, strike: float, years: float, vol: float, option_type: str, rate: float) -> tuple[float, float]:
    years = max(float(years), 1 / 365)
    vol = max(float(vol), 0.01)
    rate = float(rate)
    d1 = (math.log(spot / strike) + (rate + 0.5 * vol**2) * years) / (vol * math.sqrt(years))
    d2 = d1 - vol * math.sqrt(years)
    if option_type == 'C':
        price = spot * norm_cdf(d1) - strike * math.exp(-rate * years) * norm_cdf(d2)
        delta = norm_cdf(d1)
    else:
        price = strike * math.exp(-rate * years) * norm_cdf(-d2) - spot * norm_cdf(-d1)
        delta = norm_cdf(d1) - 1.0
    return max(price, 0.01), delta


def size_bucket(contracts: int) -> str:
    if contracts == 1:
        return '1'
    if contracts <= 5:
        return '2-5'
    if contracts <= 20:
        return '6-20'
    if contracts <= 99:
        return '21-99'
    if contracts <= 199:
        return '100-199'
    return '200-500'

In [ ]:
def generate_trades(n_trades: int = N_TRADES) -> pd.DataFrame:
    bar_idx = rng.choice(market.index.to_numpy(), size=n_trades, replace=True, p=bar_weights)
    maturities = rng.choice(MATURITY_BUCKETS.index, size=n_trades, p=MATURITY_BUCKETS['weight'].to_numpy())
    money_buckets = rng.choice(MONEYNESS_BUCKETS.index, size=n_trades, p=MONEYNESS_BUCKETS['weight'].to_numpy())
    option_types = rng.choice(['C', 'P'], size=n_trades)
    sides = rng.choice(['Buy', 'Sell'], size=n_trades)
    sampled_sizes = rng.choice(contract_size_pool, size=n_trades, replace=True)

    records = []
    for i, row_idx in enumerate(bar_idx):
        row = market.loc[row_idx]
        trade_date = pd.Timestamp(row['trade_date'])
        timestamp_et = pd.Timestamp(row['timestamp_et'])
        spot = float(row['spx_price'])
        expiry = choose_expiry(trade_date, maturities[i])
        dte = int((expiry - trade_date).days)

        money_spec = MONEYNESS_BUCKETS.loc[money_buckets[i]]
        strike_ratio = float(rng.uniform(money_spec.low, money_spec.high))
        increment = strike_increment(spot)
        strike = max(increment, round((spot * strike_ratio) / increment) * increment)

        vol = float(row['bvol_1m'])
        rate = float(row['sofr_1m_rate'])
        option_price, delta = black_scholes(spot, strike, dte / 365.0, vol, option_types[i], rate)
        contracts = int(np.clip(sampled_sizes[i], 1, 500))
        side_sign = 1 if sides[i] == 'Buy' else -1
        signed_delta_contracts = side_sign * delta * contracts * CONTRACT_MULTIPLIER
        dollar_delta = delta * contracts * CONTRACT_MULTIPLIER * spot

        records.append({
            'trade_id': f'T{i + 1:07d}',
            'timestamp_raw': row['timestamp_raw'],
            'timestamp_et': timestamp_et,
            'trade_date': trade_date,
            'symbol': 'SPX',
            'expiry': expiry,
            'dte': dte,
            'maturity_bucket': maturities[i],
            'option_type': option_types[i],
            'strike': strike,
            'spot': round(spot, 4),
            'strike_over_spot': round(strike / spot, 6),
            'moneyness_bucket': money_buckets[i],
            'side': sides[i],
            'contracts': contracts,
            'size_bucket': size_bucket(contracts),
            'time_segment': time_segment(timestamp_et),
            'bvol_1m': round(vol, 6),
            'sofr_1m_rate': round(rate, 8),
            'implied_vol': round(vol, 6),
            'option_price': round(option_price, 4),
            'delta': round(delta, 6),
            'dollar_delta': round(dollar_delta, 2),
            'signed_delta_contracts': round(signed_delta_contracts, 4),
            'signed_delta_notional': round(signed_delta_contracts * spot, 2),
            'spy_price': round(float(row['spy_price']), 4),
            'spy_volume_5m': float(row['spy_volume_5m']),
            'spy_adv_20d': float(row['spy_adv_20d']),
            'es1_price': round(float(row['es1_price']), 4),
            'es1_volume_5m': float(row['es1_volume_5m']),
            'es1_adv_20d': float(row['es1_adv_20d']),
            'sampling_weight': float(MATURITY_BUCKETS.loc[maturities[i], 'weight'] * MONEYNESS_BUCKETS.loc[money_buckets[i], 'weight'] * 0.5 * 0.5),
        })

    trades = pd.DataFrame(records).sort_values(['timestamp_et', 'trade_id']).reset_index(drop=True)
    trades['option_line'] = (
        trades['symbol'] + '_' + trades['expiry'].dt.strftime('%Y%m%d') + '_' +
        trades['option_type'] + '_' + trades['strike'].map(lambda x: f'{x:g}')
    )
    line_volume = trades.groupby('option_line')['contracts'].transform('sum')
    trades['line_contract_volume'] = line_volume
    trades['line_volume_share'] = line_volume / trades['contracts'].sum()
    return trades


trades = generate_trades()
trades.head()

## 5. Validation and Output

These checks ensure the simulated tape respects the project scope, uses cleaned BVOL/SOFR inputs, and remains compatible with downstream hedging notebooks.

In [ ]:
assert len(trades) == N_TRADES
assert trades['trade_date'].min() >= START_DATE
assert trades['trade_date'].max() <= END_DATE
assert (trades['expiry'] > trades['trade_date']).all()
assert trades['expiry'].map(lambda x: x.weekday() == 4 and 15 <= x.day <= 21).all()
assert trades['timestamp_et'].dt.time.min() >= pd.Timestamp('09:30').time()
assert trades['timestamp_et'].dt.time.max() <= pd.Timestamp('16:00').time()
assert trades['bvol_1m'].notna().all()
assert trades['sofr_1m_rate'].notna().all()
assert (trades['bvol_1m'] > 0).all()
assert (trades['sofr_1m_rate'] > 0).all()

summary = {
    'rows': len(trades),
    'date_range': (trades.trade_date.min().date(), trades.trade_date.max().date()),
    'unique_trade_dates': int(trades.trade_date.nunique()),
    'unique_option_lines': int(trades.option_line.nunique()),
    'buy_trade_share': round((trades.side == 'Buy').mean(), 4),
    'call_trade_share': round((trades.option_type == 'C').mean(), 4),
    'mean_contracts': round(trades.contracts.mean(), 2),
    'median_contracts': float(trades.contracts.median()),
    'max_contracts': int(trades.contracts.max()),
    'mean_bvol_1m': round(trades.bvol_1m.mean(), 6),
    'mean_sofr_1m_rate': round(trades.sofr_1m_rate.mean(), 6),
    'net_signed_delta_contracts': round(trades.signed_delta_contracts.sum(), 2),
}
summary

In [ ]:
distribution_checks = {
    'maturity_trade_share': trades['maturity_bucket'].value_counts(normalize=True),
    'moneyness_trade_share': trades['moneyness_bucket'].value_counts(normalize=True),
    'side_trade_share': trades['side'].value_counts(normalize=True),
    'option_type_trade_share': trades['option_type'].value_counts(normalize=True),
    'time_segment_trade_share': trades['time_segment'].value_counts(normalize=True),
    'size_bucket_trade_share': trades['size_bucket'].value_counts(normalize=True),
}

for name, values in distribution_checks.items():
    print(f'\n{name}')
    print(values.round(4).to_string())

In [ ]:
trades.to_csv(OUTPUT_TRADES, index=False)
print(f'Wrote {len(trades):,} rows to {OUTPUT_TRADES.relative_to(PROJECT_ROOT)}')
trades.sample(10, random_state=SEED).sort_values('timestamp_et')

## Interpretation and Limitations

- The output is a synthetic SPX option execution tape, not reconstructed historical JPM trade flow.
- Cleaned `1M BVOL` and `USOSFRA Curncy` are used directly in Black-Scholes. The notebook assumes a flat 1M volatility/rate input across all simulated expiries.
- BVOL missing values are filled after reversing into chronological order, using the mean of the previous and next available values.
- SPY and ES hedge-instrument fields are included for later cost and hedge comparison work. They are not used to decide trade arrivals in this notebook except for SPY volume weighting of intraday arrival times.
- Contract sizes use accessible Databento option trade sizes as a proxy because proprietary option execution sizes are unavailable.
- The generated tape is designed to feed static-band, TWAP/VWAP, execution-cost, and volatility-scaled delta-risk backtests described in the project documents.